# Week 4: Debugging

In [63]:
import io
import pandas as pd
import numpy as np

CSV = """site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
SB-01,2024-01-15,12.4,"1,240",0.82,good
SB-01,2024-02-15,13.1,"1,180",0.79,good
SB-01,2024-03-15,15.8,"2,050",1.21,good
SB-01,2024-04-15,18.2,"1,620",0.94,good
sb-01,2024-05-15,21.5,"980",0.66,good
SB-01 ,2024-06-15,-999,"540",0.51,suspect
SB-02,2024-01-15,11.8,"3,410",1.88,good
SB-02,2024-02-15,12.3,"3,290",1.92,good
SB-02,2024-03-15,14.9,"4,870",2.41,good
SB-02,2024-04-15,17.6,"3,950",2.09,estimated
SB-02,2024-05-15,20.9,"2,240",1.44,good
sb-02,2024-06-15,-999,"1,130",1.02,suspect
SB-03,2024-01-15,10.2,"620",<0.05,good
SB-03,2024-02-15,10.9,"590",<0.05,good
SB-03,2024-03-15,13.4,"1,040",0.33,good
SB-03,2024-04-15,16.1,"870",0.28,good
SB-03 ,2024-05-15,19.3,"410",<0.05,good
SB-03,2024-06-15,-999,"180",<0.05,suspect
"""

stream = pd.read_csv(io.StringIO(CSV))

In [64]:
stream

,site_id,date,water_temp_c,discharge_cfs,nitrate_mg_l,qc_flag
0,SB-01,2024-01-15,12.4,"1,240",0.82,good
1,SB-01,2024-02-15,13.1,"1,180",0.79,good
2,SB-01,2024-03-15,15.8,"2,050",1.21,good
3,SB-01,2024-04-15,18.2,"1,620",0.94,good
4,sb-01,2024-05-15,21.5,980,0.66,good
5,SB-01,2024-06-15,-999.0,540,0.51,suspect
6,SB-02,2024-01-15,11.8,"3,410",1.88,good
7,SB-02,2024-02-15,12.3,"3,290",1.92,good
8,SB-02,2024-03-15,14.9,"4,870",2.41,good
9,SB-02,2024-04-15,17.6,"3,950",2.09,estimated


How many monitoring sites are in this dataset?
    There are 3 monitoring sites.
What is a plausible range for stream water temperature in degrees Celsius?
    10-22 ºC
Roughly what mean water temperature do you expect across all measurements?
    15 ºC

In [65]:
stream['discharge_cfs'] * 0.0283168

TypeError: can't multiply sequence by non-int of type 'float'

In [66]:
stream.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   site_id        18 non-null     object 
 1   date           18 non-null     object 
 2   water_temp_c   18 non-null     float64
 3   discharge_cfs  18 non-null     object 
 4   nitrate_mg_l   18 non-null     object 
 5   qc_flag        18 non-null     object 
dtypes: float64(1), object(5)
memory usage: 996.0+ bytes


In [67]:
# Remove commas and change data type from float into an integer
stream['discharge_cfs'].str.replace(',', '').astype('int64')

# Reassign the modified discharge_csf column into the original column
stream['discharge_cfs'] = stream['discharge_cfs'].str.replace(',', '').astype('int64')


In [68]:
# Check if the data type for the discharge_csf column updated
stream.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   site_id        18 non-null     object 
 1   date           18 non-null     object 
 2   water_temp_c   18 non-null     float64
 3   discharge_cfs  18 non-null     int64  
 4   nitrate_mg_l   18 non-null     object 
 5   qc_flag        18 non-null     object 
dtypes: float64(1), int64(1), object(4)
memory usage: 996.0+ bytes


In [69]:
# Re run the original command
stream['discharge_cms'] = stream['discharge_cfs'] * 0.0283168

In [70]:
stream.dtypes

site_id           object
date              object
water_temp_c     float64
discharge_cfs      int64
nitrate_mg_l      object
qc_flag           object
discharge_cms    float64
dtype: object

In [ ]:
stream['discharge_cfs'].iloc[0] * 0.0283168   # same TypeError, one value
stream['discharge_cfs'].iloc[0]              # '1,240'

In [27]:
stream['discharge_cfs'].unique()

array(['1,240', '1,180', '2,050', '1,620', '980', '540', '3,410', '3,290',
       '4,870', '3,950', '2,240', '1,130', '620', '590', '1,040', '870',
       '410', '180'], dtype=object)

In [48]:
discharge_cms = (stream['discharge_cfs']
                    .str.replace(',', '', regex=False)
                    .astype(float)) * 0.0283168

discharge_cms.head(3)

0    35.112832
1    33.413824
2    58.049440
Name: discharge_cfs, dtype: float64

In [71]:
assert pd.api.types.is_numeric_dtype(discharge_cms), 'Discharge failed to convert to numeric'

Before we run `stream['site_id'].strip()`, this will remove any excess spaces before and after the name


In [72]:
# This does not work becuase there is no strip() method. It is .str.strip()
stream['site_id'].strip()

AttributeError: 'Series' object has no attribute 'strip'

In [73]:
# Use .str first! But this is still not our expected number of sites
stream['site_id'].str.strip().value_counts()

site_id
SB-03    6
SB-01    5
SB-02    5
sb-01    1
sb-02    1
Name: count, dtype: int64

In [74]:
stream['site_id'].str.strip().str.lower().value_counts()

site_id
sb-01    6
sb-02    6
sb-03    6
Name: count, dtype: int64

In [75]:
stream['site_id'] = stream['site_id'].str.strip().str.lower()
stream['site_id']

0     sb-01
1     sb-01
2     sb-01
3     sb-01
4     sb-01
5     sb-01
6     sb-02
7     sb-02
8     sb-02
9     sb-02
10    sb-02
11    sb-02
12    sb-03
13    sb-03
14    sb-03
15    sb-03
16    sb-03
17    sb-03
Name: site_id, dtype: object

In [76]:
# Convert the nitrate_mg_l into a numeric value
stream['nitrate_mg_l'].astype(float)

ValueError: could not convert string to float: '<0.05'

In [77]:
# Inspect
stream['nitrate_mg_l'].value_counts()

nitrate_mg_l
<0.05    4
0.82     1
0.79     1
1.21     1
0.94     1
0.66     1
0.51     1
1.88     1
1.92     1
2.41     1
2.09     1
1.44     1
1.02     1
0.33     1
0.28     1
Name: count, dtype: int64

In [78]:
stream['nitrate_mg_l'] = stream['nitrate_mg_l'].str.replace('<', '').astype(float)

In [79]:
stream.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   site_id        18 non-null     object 
 1   date           18 non-null     object 
 2   water_temp_c   18 non-null     float64
 3   discharge_cfs  18 non-null     int64  
 4   nitrate_mg_l   18 non-null     float64
 5   qc_flag        18 non-null     object 
 6   discharge_cms  18 non-null     float64
dtypes: float64(3), int64(1), object(3)
memory usage: 1.1+ KB


In [80]:
# Want to find the mean of the temperature column
stream['temperature'].mean()

KeyError: 'temperature'

The error appears because there is no column in streams called `temperature`. Inspect the data. 

In [81]:
print(stream.columns)
stream.info()

Index(['site_id', 'date', 'water_temp_c', 'discharge_cfs', 'nitrate_mg_l',
       'qc_flag', 'discharge_cms'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   site_id        18 non-null     object 
 1   date           18 non-null     object 
 2   water_temp_c   18 non-null     float64
 3   discharge_cfs  18 non-null     int64  
 4   nitrate_mg_l   18 non-null     float64
 5   qc_flag        18 non-null     object 
 6   discharge_cms  18 non-null     float64
dtypes: float64(3), int64(1), object(3)
memory usage: 1.1+ KB


In [89]:
# Now that we know the name (and we know that it's a numeric value),
#  we can try to calcualate the mean of the temperature column
stream['water_temp_c'].mean()

-153.8111111111111

This works, but ... the value -153.8111 does not seem right. The output is large and negative because it uses the -999.00 values when calculating the mean.

In [86]:
# Use dropna = False to double check there is no other NAs (would be useful if data is massive)
# This shows that the NAs are marked as -999.0
stream['water_temp_c'].value_counts(dropna= False)

water_temp_c
-999.0    3
 12.4     1
 13.1     1
 15.8     1
 18.2     1
 21.5     1
 11.8     1
 12.3     1
 14.9     1
 17.6     1
 20.9     1
 10.2     1
 10.9     1
 13.4     1
 16.1     1
 19.3     1
Name: count, dtype: int64

In [90]:
stream['water_temp_c']= stream['water_temp_c'].replace(-999.0, np.nan)

In [91]:
stream['water_temp_c'].mean()

15.226666666666668

In [87]:
site_means = stream.groupby('site_id')['water_temp_c'].mean()
stream['site_mean'] = site_means
stream[['site_id', 'water_temp_c', 'site_mean']].head()

,site_id,water_temp_c,site_mean
0,sb-01,12.4,NaN
1,sb-01,13.1,NaN
2,sb-01,15.8,NaN
3,sb-01,18.2,NaN
4,sb-01,21.5,NaN


In [88]:
site_means = stream.groupby('site_id')['water_temp_c'].mean()
stream['site_mean'] = site_means
stream[['site_id', 'water_temp_c', 'site_mean']].head()

,site_id,water_temp_c,site_mean
0,sb-01,12.4,NaN
1,sb-01,13.1,NaN
2,sb-01,15.8,NaN
3,sb-01,18.2,NaN
4,sb-01,21.5,NaN
